# Logistic Bayesian Regression - Variational Inference (Polya-Gamma)

Binary logistic regression: y_i ~ Bernoulli(sigma(x_i^T beta)), beta_true = (-1.0, 2.0), n=200.

ELBO: Jaakkola-Jordan (1996) lower bound at optimal xi_i*.
Reference sampler: Polya-Gamma Gibbs (Polson, Scott & Windle 2013).

**Five stages:**
1. Data generation
2. ELBO evaluation + gradient verification
3. Optimisation methods (CAVI, gradient ascent, Newton, BFGS)
4. Reference sampler (Polya-Gamma Gibbs) with ESS diagnostics
5. Diagnostics and comparison (std ratios, timing, cost-per-ESS)

In [1]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.special import expit
from scipy.optimize import minimize
import time

ROOT   = Path("..").resolve()
FIGS   = ROOT / "figs"
DATA   = ROOT / "results" / "data"
TABLES = ROOT / "results" / "tables"
for d in [FIGS, DATA, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 82171165
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi":150,"axes.grid":True,"grid.alpha":0.3,"font.size":10})

def save_fig(fig, name):
    path = FIGS / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved: {path.name}")
    plt.close(fig)

print("Setup complete. ROOT =", ROOT)


Setup complete. ROOT = D:\github\ENEL445


---
## Stage 1 - Data Generation

Generate n=200 binary observations. Design matrix X=[1, x], x ~ Uniform(-2,2). beta_true=(-1.0, 2.0).

In [2]:
BETA_TRUE = np.array([-1.0, 2.0])
N = 200
np.random.seed(SEED)
x    = np.random.uniform(-2.0, 2.0, N)
X    = np.column_stack([np.ones(N), x])
psi  = X @ BETA_TRUE
prob = expit(psi)
y    = np.random.binomial(1, prob)
print(f"n={N}, p=2, beta_true={BETA_TRUE}")
print(f"y=1: {y.sum()},  y=0: {(1-y).sum()}")
pl.DataFrame({"x":x,"y":y.astype(float),"prob_true":prob}).write_parquet(DATA/"logistic_data.parquet")
print("  Saved: logistic_data.parquet")


n=200, p=2, beta_true=[-1.  2.]
y=1: 64,  y=0: 136
  Saved: logistic_data.parquet


In [3]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(x[y==0], y[y==0], s=15, alpha=0.5, color="steelblue", label="y=0")
ax.scatter(x[y==1], y[y==1], s=15, alpha=0.5, color="tomato",    label="y=1")
xs = np.linspace(-2.1, 2.1, 300)
ax.plot(xs, expit(BETA_TRUE[0]+BETA_TRUE[1]*xs), "k-", lw=1.5, label="true p(y=1|x)")
ax.set_xlabel("x"); ax.set_ylabel("y / P(y=1|x)")
ax.set_title("Logistic data - true sigmoid curve")
ax.legend(fontsize=8); fig.tight_layout()
save_fig(fig, "logistic_data_scatter.png")


  Saved: logistic_data_scatter.png


---
## Stage 2 - ELBO Evaluation and Gradient Verification

Variational family: q(beta) = N(mu, Sigma), Sigma=LL^T (Cholesky). Unconstrained theta=(mu0,mu1,log(l11),l21,log(l22)).

Jaakkola-Jordan lower bound at optimal xi_i* = sqrt((x_i^T mu)^2 + x_i^T Sigma x_i):
L(theta) = sum_i [(y_i-0.5) x_i^T mu - log2 - log cosh(xi_i*/2)] - KL(q||p)

Prior: beta ~ N(0, 10*I).

In [4]:
P = 2; M0 = np.zeros(P); V0 = 10.*np.eye(P); V0INV = np.linalg.inv(V0)
LOG_DET_V0 = np.log(np.linalg.det(V0))

def unpack(theta):
    mu = theta[:2]
    L  = np.array([[np.exp(theta[2]), 0.],
                   [theta[3],         np.exp(theta[4])]])
    return mu, L, L @ L.T

def pack(mu, L):
    return np.array([mu[0], mu[1], np.log(L[0,0]), L[1,0], np.log(L[1,1])])

def elbo(theta):
    mu, L, Sigma = unpack(theta)
    psi = X @ mu
    s2  = np.einsum("ij,jk,ik->i", X, Sigma, X)
    xi  = np.maximum(np.sqrt(psi**2 + s2), 1e-10)
    lik = np.sum((y - 0.5)*psi - np.log(2.) - np.log(np.cosh(xi/2.)))
    log_det_S = 2.*(np.log(L[0,0]) + np.log(L[1,1]))
    kl = .5*(LOG_DET_V0 - log_det_S - P
             + np.trace(V0INV @ Sigma)
             + (mu - M0) @ V0INV @ (mu - M0))
    return lik - kl

L0 = 0.1*np.eye(P); theta0 = pack(np.zeros(P), L0)
print(f"theta0       = {theta0}")
print(f"ELBO(theta0) = {elbo(theta0):.4f}")


theta0       = [ 0.          0.         -2.30258509  0.         -2.30258509]
ELBO(theta0) = -145.1074


In [5]:
def elbo_grad(theta):
    mu, L, Sigma = unpack(theta)
    psi = X @ mu
    s2  = np.einsum("ij,jk,ik->i", X, Sigma, X)
    xi  = np.maximum(np.sqrt(psi**2 + s2), 1e-10)
    th2 = np.tanh(xi/2.)
    g_mu = (np.sum((y-0.5)[:,None]*X, axis=0)
            - np.sum((th2/(2.*xi))[:,None]*(X*(psi/xi)[:,None]), axis=0)
            - V0INV @ (mu - M0))
    LTX     = X @ L
    factor  = (th2/(2.*xi))[:,None,None]
    dlik_dL = -np.sum(factor * X[:,:,None] * LTX[:,None,:], axis=0)
    dkl_dL  = (np.linalg.inv(Sigma) - V0INV) @ L
    dL      = dlik_dL + dkl_dL
    return np.array([g_mu[0], g_mu[1],
                     dL[0,0]*L[0,0], dL[1,0], dL[1,1]*L[1,1]])

EPS=1e-5; g_an=elbo_grad(theta0); g_fd=np.zeros(5)
for k in range(5):
    tp,tm=theta0.copy(),theta0.copy(); tp[k]+=EPS; tm[k]-=EPS
    g_fd[k]=(elbo(tp)-elbo(tm))/(2*EPS)
print("Gradient check at theta0:")
labs = ["dmu0","dmu1","dtl11","dl21","dtl22"]
for k,lab in enumerate(labs):
    rel = abs(g_an[k]-g_fd[k])/(abs(g_fd[k])+1e-12)
    print(f"  {lab}: analytical={g_an[k]:+.5f}  FD={g_fd[k]:+.5f}  rel_err={rel:.2e}")


Gradient check at theta0:
  dmu0: analytical=-36.00000  FD=-36.00000  rel_err=1.62e-12
  dmu1: analytical=+75.48684  FD=+75.48684  rel_err=6.73e-12
  dtl11: analytical=+0.49995  FD=+0.49995  rel_err=9.28e-10
  dl21: analytical=+0.39875  FD=+0.39875  rel_err=3.51e-09
  dtl22: analytical=+0.36107  FD=+0.36107  rel_err=3.57e-11


In [6]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x_pos = np.arange(5); w = 0.35
labs = ["dmu0","dmu1","dtl11","dl21","dtl22"]
ax.bar(x_pos-w/2, g_an, w, label="Analytical", color="steelblue")
ax.bar(x_pos+w/2, g_fd, w, label="Finite diff", color="tomato", alpha=0.7)
ax.set_xticks(x_pos); ax.set_xticklabels(labs)
ax.set_ylabel("Gradient"); ax.set_title("Gradient verification at theta_0")
ax.legend(fontsize=8); fig.tight_layout()
save_fig(fig, "logistic_elbo_gradient_check.png")


  Saved: logistic_elbo_gradient_check.png


In [7]:
mu1_vals = np.linspace(-3, 5, 120)
elbo_slice = [elbo(np.array([theta0[0], v, theta0[2], theta0[3], theta0[4]])) for v in mu1_vals]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(mu1_vals, elbo_slice, "b-")
ax.axvline(BETA_TRUE[1], color="r", ls="--", lw=1.5, label="beta_1 true")
ax.set_xlabel("mu_1"); ax.set_ylabel("ELBO"); ax.set_title("ELBO slice along mu_1")
ax.legend(); fig.tight_layout()
save_fig(fig, "logistic_elbo_landscape.png")


  Saved: logistic_elbo_landscape.png


---
## Stage 3 - Optimisation Methods

Four methods applied to maximise L(theta):

| Method | Entry condition | Search direction | Step-size rule |
|--------|----------------|------------------|----------------|
| CAVI | - | Closed-form coordinate update | - |
| Gradient ascent | grad^T d > 0 | grad L | Armijo backtracking |
| Newton | grad^T d > 0 | -H_reg^-1 grad L | Armijo backtracking |
| BFGS | Wolfe curvature | Quasi-Newton | Wolfe conditions |

Exit conditions (any): gradient norm < 1e-8, relative ELBO change < 1e-8, parameter change < 1e-8, or max iterations.

In [8]:
def run_cavi(max_iter=500, tol=1e-8):
    mu = np.zeros(P); Sigma = np.eye(P)
    elbo_hist = []
    t0 = time.perf_counter()
    for it in range(max_iter):
        s2     = np.einsum("ij,jk,ik->i", X, Sigma, X)
        psi_mu = X @ mu
        xi     = np.maximum(np.sqrt(psi_mu**2 + s2), 1e-10)
        lam    = np.tanh(xi/2.) / (4.*xi)
        Sigma  = np.linalg.inv(V0INV + 2.*(X.T * lam) @ X)
        mu     = Sigma @ (V0INV @ M0 + X.T @ (y - 0.5))
        L_cur  = np.linalg.cholesky(Sigma)
        elbo_hist.append(elbo(pack(mu, L_cur)))
        if it > 0 and abs(elbo_hist[-1] - elbo_hist[-2]) < tol:
            break
    elapsed = (time.perf_counter() - t0)*1000
    print(f"CAVI:        {it+1:4d} iters  ELBO={elbo_hist[-1]:.5f}  {elapsed:.1f} ms")
    return mu, Sigma, np.array(elbo_hist), elapsed

mu_cavi, Sig_cavi, elbo_cavi, t_cavi_ms = run_cavi()
L_cavi = np.linalg.cholesky(Sig_cavi)
theta_cavi = pack(mu_cavi, L_cavi)
print(f"  mu={mu_cavi.round(4)}  std={np.sqrt(np.diag(Sig_cavi)).round(4)}")


CAVI:          41 iters  ELBO=-71.24090  6.0 ms
  mu=[-1.4995  2.3389]  std=[0.1732 0.1683]


In [9]:
def run_gradient_ascent(theta_init, max_iter=2000, alpha0=0.5, rho=0.5, c1=1e-4, tol=1e-8):
    theta = theta_init.copy()
    elbo_hist, step_hist = [], []
    t0 = time.perf_counter()
    for it in range(max_iter):
        e = elbo(theta); g = elbo_grad(theta)
        elbo_hist.append(e)
        alpha = alpha0
        for _ in range(60):
            if elbo(theta + alpha*g) >= e + c1*alpha*np.dot(g,g):
                break
            alpha *= rho
        step_hist.append(alpha)
        theta_new = theta + alpha*g
        if np.linalg.norm(g) < tol:
            theta = theta_new; break
        theta = theta_new
    elapsed = (time.perf_counter() - t0)*1000
    print(f"Grad ascent: {it+1:4d} iters  ELBO={elbo_hist[-1]:.5f}  {elapsed:.1f} ms")
    return theta, np.array(elbo_hist), np.array(step_hist), elapsed

theta_ga, elbo_ga, step_ga, t_ga_ms = run_gradient_ascent(theta0)
mu_ga, _, Sig_ga = unpack(theta_ga)
print(f"  mu={mu_ga.round(4)}  std={np.sqrt(np.diag(Sig_ga)).round(4)}")


Grad ascent: 2000 iters  ELBO=-89.16625  7078.4 ms
  mu=[-2.25    4.7179]  std=[0.1032 0.1053]


In [10]:
def fd_hessian(theta, eps=1e-4):
    n = len(theta); H = np.zeros((n,n))
    for i in range(n):
        tp,tm = theta.copy(),theta.copy(); tp[i]+=eps; tm[i]-=eps
        H[:,i] = (elbo_grad(tp) - elbo_grad(tm))/(2*eps)
    return (H+H.T)/2

def run_newton(theta_init, max_iter=100, lam_init=1e-3, tol=1e-8):
    theta = theta_init.copy()
    elbo_hist, cond_hist, step_hist = [], [], []
    t0 = time.perf_counter()
    for it in range(max_iter):
        e = elbo(theta); g = elbo_grad(theta); H = fd_hessian(theta)
        elbo_hist.append(e)
        lam = lam_init
        for _ in range(30):
            try:
                d = -np.linalg.solve(H - lam*np.eye(len(theta)), g)
            except np.linalg.LinAlgError:
                d = g; break
            if np.dot(g,d) > 0:
                break
            lam *= 10
            if lam > 1e6: d = g; break
        cond_hist.append(np.linalg.cond(H - lam*np.eye(len(theta))))
        alpha = 1.0
        for _ in range(50):
            if elbo(theta + alpha*d) >= e + 1e-4*alpha*np.dot(g,d):
                break
            alpha *= 0.5
        step_hist.append(alpha)
        theta_new = theta + alpha*d
        if np.linalg.norm(theta_new-theta) < tol:
            theta = theta_new; break
        theta = theta_new
    elapsed = (time.perf_counter() - t0)*1000
    print(f"Newton:      {it+1:4d} iters  ELBO={elbo_hist[-1]:.5f}  {elapsed:.1f} ms")
    return theta, np.array(elbo_hist), np.array(cond_hist), np.array(step_hist), elapsed

theta_newton, elbo_newton, cond_newton, step_newton, t_newton_ms = run_newton(theta0)
mu_newton, _, Sig_newton = unpack(theta_newton)
print(f"  mu={mu_newton.round(4)}  std={np.sqrt(np.diag(Sig_newton)).round(4)}")


Newton:         4 iters  ELBO=-84.77403  9.5 ms
  mu=[-2.2651  3.5671]  std=[0.6814 0.7189]


In [11]:
t0 = time.perf_counter()
res_bfgs = minimize(lambda t: -elbo(t), theta0,
                    jac=lambda t: -elbo_grad(t),
                    method="BFGS",
                    options={"maxiter":2000,"gtol":1e-8})
t_bfgs_ms = (time.perf_counter()-t0)*1000
theta_bfgs = res_bfgs.x
mu_bfgs, _, Sig_bfgs = unpack(theta_bfgs)
print(f"BFGS:        {res_bfgs.nit:4d} iters  ELBO={-res_bfgs.fun:.5f}  {t_bfgs_ms:.1f} ms  success={res_bfgs.success}")
print(f"  mu={mu_bfgs.round(4)}  std={np.sqrt(np.diag(Sig_bfgs)).round(4)}")


BFGS:           2 iters  ELBO=-89.17604  10.8 ms  success=False
  mu=[-0.6952  0.857 ]  std=[0.1017 0.1021]


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(elbo_cavi,   lw=1.5, color="steelblue",  label="CAVI")
ax.plot(elbo_ga,     lw=1.5, color="darkorange",  label="Grad ascent")
ax.plot(elbo_newton, lw=1.5, color="forestgreen", label="Newton")
ax.set_xlabel("Iteration"); ax.set_ylabel("ELBO")
ax.set_title("ELBO convergence - logistic case"); ax.legend()
ax = axes[1]
ax.plot(np.maximum(1e-12, np.abs(np.diff(elbo_cavi))),   lw=1.5, color="steelblue",  label="CAVI")
ax.plot(np.maximum(1e-12, np.abs(np.diff(elbo_ga))),     lw=1.5, color="darkorange",  label="Grad ascent")
ax.plot(np.maximum(1e-12, np.abs(np.diff(elbo_newton))), lw=1.5, color="forestgreen", label="Newton")
ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_ylabel("|delta ELBO|")
ax.set_title("Convergence rate (log scale)"); ax.legend()
fig.tight_layout()
save_fig(fig, "logistic_elbo_convergence.png")


  Saved: logistic_elbo_convergence.png


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(step_ga,     lw=1.5, color="darkorange")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Step size (log)")
axes[0].set_title("Gradient ascent - Armijo step size")
axes[1].semilogy(step_newton, lw=1.5, color="forestgreen")
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Step size (log)")
axes[1].set_title("Newton - Armijo step size")
fig.tight_layout()
save_fig(fig, "logistic_gradient_stepsize.png")


  Saved: logistic_gradient_stepsize.png


In [14]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(cond_newton, lw=1.5, color="forestgreen")
ax.set_xlabel("Iteration"); ax.set_ylabel("Condition number (log)")
ax.set_title("Newton - regularised Hessian condition number")
fig.tight_layout()
save_fig(fig, "logistic_newton_condition.png")


  Saved: logistic_newton_condition.png


In [15]:
vi_methods = {
    "CAVI":       (mu_cavi,   Sig_cavi,   elbo_cavi[-1],   t_cavi_ms),
    "GradAscent": (mu_ga,     Sig_ga,     elbo_ga[-1],     t_ga_ms),
    "Newton":     (mu_newton, Sig_newton, elbo_newton[-1], t_newton_ms),
    "BFGS":       (mu_bfgs,   Sig_bfgs,   -res_bfgs.fun,   t_bfgs_ms),
}
rows = []
for name,(mu_v,Sig_v,fe,t_ms) in vi_methods.items():
    rows.append({"method":name,"mu0":mu_v[0],"mu1":mu_v[1],
                 "std0":np.sqrt(Sig_v[0,0]),"std1":np.sqrt(Sig_v[1,1]),
                 "final_elbo":fe,"time_ms":t_ms})
df_opt = pl.DataFrame(rows)
df_opt.write_parquet(DATA/"logistic_opt.parquet")
print(df_opt)


shape: (4, 7)
┌───────────┬───────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ method    ┆ mu0       ┆ mu1      ┆ std0     ┆ std1     ┆ final_el ┆ time_ms  │
│ ---       ┆ ---       ┆ ---      ┆ ---      ┆ ---      ┆ bo       ┆ ---      │
│ str       ┆ f64       ┆ f64      ┆ f64      ┆ f64      ┆ ---      ┆ f64      │
│           ┆           ┆          ┆          ┆          ┆ f64      ┆          │
╞═══════════╪═══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ CAVI      ┆ -1.49948  ┆ 2.338857 ┆ 0.173215 ┆ 0.168328 ┆ -71.2409 ┆ 6.0415   │
│           ┆           ┆          ┆          ┆          ┆ 05       ┆          │
│ GradAscen ┆ -2.25     ┆ 4.717928 ┆ 0.103174 ┆ 0.105275 ┆ -89.1662 ┆ 7078.379 │
│ t         ┆           ┆          ┆          ┆          ┆ 54       ┆ 3        │
│ Newton    ┆ -2.265147 ┆ 3.5671   ┆ 0.681394 ┆ 0.718874 ┆ -84.7740 ┆ 9.4735   │
│           ┆           ┆          ┆          ┆          ┆ 25       ┆          │
│ BFGS      ┆ 

---
## Stage 4 - Reference Sampler: Polya-Gamma Gibbs

Polson, Scott & Windle (2013) sec 3.1. Each iteration:
1. Draw omega_i | beta ~ PG(1, x_i^T beta)
2. Draw beta | omega, y ~ N(m_omega, V_omega) where V_omega=(X^T Omega X + B^-1)^-1, m_omega=V_omega(X^T kappa + B^-1 b), kappa_i=y_i-0.5

Prior: beta ~ N(0, 10 I). PG sampler: T=50 truncated gamma terms.
Note on timing: PG Gibbs involves 200 x N_iter gamma draws (n=200 per iteration). Timing is not directly comparable to the Gaussian Gibbs in the linear/quadratic cases.

In [16]:
def sample_pg_one(c, T=50):
    c = abs(float(c))
    k = np.arange(1, T+1, dtype=float)
    g = np.random.gamma(1., 1., T)
    return np.sum(g / ((k-0.5)**2 + (c/(2.*np.pi))**2)) / (2.*np.pi**2)

def sample_pg_vec(cvec):
    return np.array([sample_pg_one(ci) for ci in cvec])

samp = np.array([sample_pg_one(0.) for _ in range(500)])
print(f"E[PG(1,0)] mean={samp.mean():.4f}  theory=0.2500  T=50")


E[PG(1,0)] mean=0.2466  theory=0.2500  T=50


In [17]:
N_WARMUP  = 500
N_SAMPLES = 2000
kappa     = y - 0.5
V0inv_b   = V0INV @ M0

samples_beta = np.zeros((N_SAMPLES, P))
beta_cur     = np.zeros(P)

print(f"Running {N_WARMUP} warmup + {N_SAMPLES} Gibbs iterations (PG, T=50)...")
t0 = time.perf_counter()

for i in range(N_WARMUP + N_SAMPLES):
    psi_cur = X @ beta_cur
    omega   = sample_pg_vec(psi_cur)
    V_omega = np.linalg.inv((X.T * omega) @ X + V0INV)
    m_omega = V_omega @ (X.T @ kappa + V0inv_b)
    beta_cur = np.random.multivariate_normal(m_omega, V_omega)
    if i >= N_WARMUP:
        samples_beta[i-N_WARMUP] = beta_cur

t_gibbs_ms = (time.perf_counter()-t0)*1000
gibbs_mean = samples_beta.mean(axis=0)
gibbs_std  = samples_beta.std(axis=0)
print(f"Done.  time={t_gibbs_ms/1000:.1f} s")
print(f"  Posterior mean = {gibbs_mean.round(4)}")
print(f"  Posterior std  = {gibbs_std.round(4)}")
pl.DataFrame({"beta0":samples_beta[:,0],"beta1":samples_beta[:,1]}).write_parquet(DATA/"logistic_gibbs.parquet")
print("  Saved: logistic_gibbs.parquet")


Running 500 warmup + 2000 Gibbs iterations (PG, T=50)...


Done.  time=6.5 s
  Posterior mean = [-1.6161  2.5141]
  Posterior std  = [0.3144 0.371 ]
  Saved: logistic_gibbs.parquet


In [18]:
param_names = ["beta0","beta1"]
param_tex   = ["beta0", "beta1"]

for j, (pname, ptex) in enumerate(zip(param_names, param_tex)):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(samples_beta[:,j], lw=0.4, alpha=0.7, color="steelblue")
    axes[0].axhline(BETA_TRUE[j], color="r", lw=1.2, ls="--", label="true")
    axes[0].axhline(gibbs_mean[j], color="k", lw=1.0, ls=":", label="post mean")
    axes[0].set_xlabel("sample"); axes[0].set_ylabel(ptex)
    axes[0].set_title(f"Trace {ptex}"); axes[0].legend(fontsize=8)
    axes[1].hist(samples_beta[:,j], bins=60, density=True, color="steelblue", alpha=0.7)
    axes[1].axvline(BETA_TRUE[j], color="r", lw=1.5, ls="--", label="true")
    axes[1].axvline(gibbs_mean[j], color="k", lw=1.2, ls=":", label="post mean")
    axes[1].set_xlabel(ptex); axes[1].set_title(f"Posterior {ptex}")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, f"logistic_gibbs_trace_{pname}.png")


  Saved: logistic_gibbs_trace_beta0.png


  Saved: logistic_gibbs_trace_beta1.png


In [19]:
def acf(x, max_lag=50):
    x = x - x.mean()
    var = np.var(x)
    return np.array([np.mean(x[k:]*x[:len(x)-k])/var if k>0 else 1. for k in range(max_lag+1)])

for j, (pname, ptex) in enumerate(zip(param_names, param_tex)):
    ac = acf(samples_beta[:,j])
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(range(len(ac)), ac, color="steelblue", alpha=0.7)
    ax.axhline(0, color="k", lw=0.8)
    ax.axhline(1.96/np.sqrt(N_SAMPLES), color="r", ls="--", lw=1, label="95% CI")
    ax.axhline(-1.96/np.sqrt(N_SAMPLES), color="r", ls="--", lw=1)
    ax.set_xlabel("Lag"); ax.set_ylabel("ACF"); ax.set_title(f"ACF {ptex} (PG Gibbs)")
    ax.legend(fontsize=8); fig.tight_layout()
    save_fig(fig, f"logistic_gibbs_acf_{pname}.png")


  Saved: logistic_gibbs_acf_beta0.png


  Saved: logistic_gibbs_acf_beta1.png


In [20]:
def compute_ess(s1d, max_lag=200):
    ac = acf(s1d, max_lag=min(max_lag, len(s1d)//4))
    rho_sum = 0.
    for k in range(1, len(ac)):
        if ac[k] < 0: break
        rho_sum += ac[k]
    return len(s1d) / (1. + 2.*rho_sum)

ess = np.array([compute_ess(samples_beta[:,j]) for j in range(P)])
cost_per_ess = t_gibbs_ms / ess
print("ESS diagnostics (PG Gibbs):")
for j, pname in enumerate(param_names):
    print(f"  {pname}: ESS={ess[j]:.0f}/{N_SAMPLES} ({100*ess[j]/N_SAMPLES:.0f}%)  cost/ESS={cost_per_ess[j]:.2f} ms")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(param_tex, ess, color=["steelblue","tomato"])
axes[0].axhline(N_SAMPLES, color="k", ls="--", lw=1, label=f"N={N_SAMPLES}")
axes[0].set_ylabel("ESS"); axes[0].set_title("Effective sample size (PG Gibbs)")
axes[0].legend(fontsize=8)
axes[1].bar(param_tex, cost_per_ess, color=["steelblue","tomato"])
axes[1].set_ylabel("ms per effective sample")
axes[1].set_title("Cost per ESS (PG Gibbs)")
fig.tight_layout()
save_fig(fig, "logistic_gibbs_ess.png")


ESS diagnostics (PG Gibbs):
  beta0: ESS=292/2000 (15%)  cost/ESS=22.11 ms
  beta1: ESS=235/2000 (12%)  cost/ESS=27.51 ms


  Saved: logistic_gibbs_ess.png


---
## Stage 5 - Diagnostics and Comparison

Compare VI methods against the Polya-Gamma Gibbs reference.

**Note on cross-model timing:** Comparisons across linear, quadratic, and logistic notebooks are not controlled experiments. The fair comparison is VI vs. the notebook's own Gibbs reference. Cross-model timings are qualitative context only.

In [21]:
xs_p = np.linspace(-5, 6, 400)
for j, (pname, ptex) in enumerate(zip(param_names, param_tex)):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(samples_beta[:,j], bins=60, density=True, color="lightgrey", alpha=0.9, label="Gibbs (PG)", zorder=1)
    for mu_v,Sig_v,lbl,col in [
            (mu_cavi,   Sig_cavi,   "CAVI",       "steelblue"),
            (mu_ga,     Sig_ga,     "Grad ascent","darkorange"),
            (mu_bfgs,   Sig_bfgs,   "BFGS",       "purple")]:
        s = np.sqrt(Sig_v[j,j])
        ax.plot(xs_p, stats.norm.pdf(xs_p, mu_v[j], s), lw=1.8, label=lbl, color=col)
    ax.axvline(BETA_TRUE[j], color="k", ls="--", lw=1.3, label="true")
    ax.set_xlabel(ptex); ax.set_title(f"Posterior comparison {ptex}")
    ax.legend(fontsize=7); fig.tight_layout()
    save_fig(fig, f"logistic_vb_gibbs_{pname}.png")


  Saved: logistic_vb_gibbs_beta0.png


  Saved: logistic_vb_gibbs_beta1.png


In [22]:
xs_pred = np.linspace(-2.2, 2.2, 200)
X_pred  = np.column_stack([np.ones(200), xs_pred])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(x[y==0], y[y==0], s=12, alpha=0.4, color="steelblue")
ax.scatter(x[y==1], y[y==1], s=12, alpha=0.4, color="tomato")
ax.plot(xs_pred, expit(X_pred @ BETA_TRUE), "k-", lw=1.5, label="True")
for mu_v, lbl, col in [(mu_bfgs,"BFGS","purple"),(gibbs_mean,"Gibbs (PG)","grey")]:
    ax.plot(xs_pred, expit(X_pred @ mu_v), "--", lw=1.5, label=lbl, color=col)
ax.set_xlabel("x"); ax.set_ylabel("P(y=1|x)")
ax.set_title("Predictive sigmoid - true vs estimated")
ax.legend(fontsize=8); fig.tight_layout()
save_fig(fig, "logistic_predictive.png")


  Saved: logistic_predictive.png


In [23]:
sd_ratios = {}
for name,(mu_v,Sig_v,fe,t_ms) in vi_methods.items():
    sd_ratios[name] = [np.sqrt(Sig_v[j,j])/gibbs_std[j] for j in range(P)]

method_list = list(vi_methods.keys())
fig, ax = plt.subplots(figsize=(7, 4))
x_pos = np.arange(P); w = 0.18
colors = ["steelblue","darkorange","forestgreen","purple"]
for i,(name,col) in enumerate(zip(method_list,colors)):
    ax.bar(x_pos + (i-1.5)*w, sd_ratios[name], w, label=name, color=col, alpha=0.85)
ax.axhline(1.0, color="k", ls="--", lw=1.2, label="Gibbs reference")
ax.set_xticks(x_pos); ax.set_xticklabels(param_tex)
ax.set_ylabel("Std ratio (VI / Gibbs)"); ax.set_title("Posterior std ratios - logistic case")
ax.legend(fontsize=8); fig.tight_layout()
save_fig(fig, "logistic_sd_ratios.png")


  Saved: logistic_sd_ratios.png


In [24]:
fig, ax = plt.subplots(figsize=(7, 4))
for i,(name,col) in enumerate(zip(method_list,colors)):
    mu_v,Sig_v,fe,t_ms = vi_methods[name]
    errors = np.abs(mu_v - gibbs_mean)
    ax.bar(x_pos + (i-1.5)*w, errors, w, label=name, color=col, alpha=0.85)
ax.set_xticks(x_pos); ax.set_xticklabels(param_tex)
ax.set_ylabel("|VI mean - Gibbs mean|"); ax.set_title("Posterior mean errors - logistic case")
ax.legend(fontsize=8); fig.tight_layout()
save_fig(fig, "logistic_mean_errors.png")


  Saved: logistic_mean_errors.png


In [25]:
for j, (pname, ptex) in enumerate(zip(param_names, param_tex)):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(samples_beta[:,j], bins=60, density=True, color="lightgrey", alpha=0.9, label="Gibbs (PG)")
    for mu_v,Sig_v,lbl,col in [
            (mu_cavi,   Sig_cavi,   "CAVI",       "steelblue"),
            (mu_newton, Sig_newton, "Newton",     "forestgreen"),
            (mu_bfgs,   Sig_bfgs,   "BFGS",       "purple")]:
        ax.plot(xs_p, stats.norm.pdf(xs_p, mu_v[j], np.sqrt(Sig_v[j,j])),
                lw=1.8, label=lbl, color=col)
    ax.axvline(BETA_TRUE[j], color="k", ls="--", lw=1.3, label="true")
    ax.set_xlabel(ptex); ax.set_title(f"All methods - posterior {ptex}")
    ax.legend(fontsize=7); fig.tight_layout()
    save_fig(fig, f"logistic_comparison_{pname}.png")


  Saved: logistic_comparison_beta0.png


  Saved: logistic_comparison_beta1.png


In [26]:
all_labels = list(vi_methods.keys()) + ["Gibbs (PG)"]
all_times  = [vi_methods[k][3] for k in vi_methods] + [t_gibbs_ms]
all_colors = ["steelblue","darkorange","forestgreen","purple","#E7298A"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(all_labels, all_times, color=all_colors, alpha=0.85, edgecolor="white")
for bar, t_ms in zip(bars[:-1], all_times[:-1]):
    speedup = t_gibbs_ms / t_ms
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.15,
            f"{speedup:.0f}x faster", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()/2,
            f"{t_ms:.0f} ms", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
ax.text(bars[-1].get_x()+bars[-1].get_width()/2, bars[-1].get_height()/2,
        f"{t_gibbs_ms:.0f} ms", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
ax.set_yscale("log"); ax.set_ylabel("Runtime (ms, log scale)")
ax.set_title("Runtime comparison - logistic case\n(speedup vs Polya-Gamma Gibbs reference)")
ax.axhline(t_gibbs_ms, color="#E7298A", lw=1.2, ls="--", alpha=0.5, label="Gibbs runtime")
ax.legend(fontsize=9); plt.tight_layout()
save_fig(fig, "logistic_timing_comparison.png")

  Saved: logistic_timing_comparison.png


In [27]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(param_tex, cost_per_ess, color=["steelblue","tomato"], alpha=0.85)
ax.set_ylabel("ms per effective sample")
ax.set_title("PG Gibbs - cost per effective sample\n(VI methods are optimisers, not samplers)")
fig.tight_layout()
save_fig(fig, "logistic_cost_per_ess.png")

  Saved: logistic_cost_per_ess.png


In [28]:
rows = []
for name,(mu_v,Sig_v,fe,t_ms) in vi_methods.items():
    rows.append({"method":name,
                 "mu0":round(float(mu_v[0]),4),"mu1":round(float(mu_v[1]),4),
                 "std0":round(float(np.sqrt(Sig_v[0,0])),4),
                 "std1":round(float(np.sqrt(Sig_v[1,1])),4),
                 "std0_ratio":round(float(np.sqrt(Sig_v[0,0])/gibbs_std[0]),3),
                 "std1_ratio":round(float(np.sqrt(Sig_v[1,1])/gibbs_std[1]),3),
                 "final_elbo":round(float(fe),5),
                 "time_ms":round(float(t_ms),1)})
rows.append({"method":"Gibbs (PG)",
             "mu0":round(float(gibbs_mean[0]),4),"mu1":round(float(gibbs_mean[1]),4),
             "std0":round(float(gibbs_std[0]),4),"std1":round(float(gibbs_std[1]),4),
             "std0_ratio":1.000,"std1_ratio":1.000,
             "final_elbo":float("nan"),"time_ms":round(float(t_gibbs_ms),1)})
df_diag = pl.DataFrame(rows)
df_diag.write_parquet(TABLES/"logistic_diagnostics.parquet")
df_diag.write_csv(TABLES/"logistic_diagnostics.csv")
print(df_diag)


shape: (5, 9)
┌─────────┬─────────┬────────┬────────┬───┬─────────┬────────┬────────┬────────┐
│ method  ┆ mu0     ┆ mu1    ┆ std0   ┆ … ┆ std0_ra ┆ std1_r ┆ final_ ┆ time_m │
│ ---     ┆ ---     ┆ ---    ┆ ---    ┆   ┆ tio     ┆ atio   ┆ elbo   ┆ s      │
│ str     ┆ f64     ┆ f64    ┆ f64    ┆   ┆ ---     ┆ ---    ┆ ---    ┆ ---    │
│         ┆         ┆        ┆        ┆   ┆ f64     ┆ f64    ┆ f64    ┆ f64    │
╞═════════╪═════════╪════════╪════════╪═══╪═════════╪════════╪════════╪════════╡
│ CAVI    ┆ -1.4995 ┆ 2.3389 ┆ 0.1732 ┆ … ┆ 0.551   ┆ 0.454  ┆ -71.24 ┆ 6.0    │
│         ┆         ┆        ┆        ┆   ┆         ┆        ┆ 09     ┆        │
│ GradAsc ┆ -2.25   ┆ 4.7179 ┆ 0.1032 ┆ … ┆ 0.328   ┆ 0.284  ┆ -89.16 ┆ 7078.4 │
│ ent     ┆         ┆        ┆        ┆   ┆         ┆        ┆ 625    ┆        │
│ Newton  ┆ -2.2651 ┆ 3.5671 ┆ 0.6814 ┆ … ┆ 2.167   ┆ 1.938  ┆ -84.77 ┆ 9.5    │
│         ┆         ┆        ┆        ┆   ┆         ┆        ┆ 403    ┆        │
│ BFGS    ┆ -0

In [29]:
print("\nAll outputs complete.")
print("\nFigures saved:")
for f in sorted(FIGS.glob("logistic_*.png")):
    print(f"  {f.name}")
print("\nTables saved:")
for f in sorted(TABLES.glob("logistic_*")):
    print(f"  {f.name}")


All outputs complete.

Figures saved:
  logistic_comparison_beta0.png
  logistic_comparison_beta1.png
  logistic_cost_per_ess.png
  logistic_data_scatter.png
  logistic_elbo_convergence.png
  logistic_elbo_gradient_check.png
  logistic_elbo_landscape.png
  logistic_gibbs_acf_beta0.png
  logistic_gibbs_acf_beta1.png
  logistic_gibbs_ess.png
  logistic_gibbs_trace_beta0.png
  logistic_gibbs_trace_beta1.png
  logistic_gradient_stepsize.png
  logistic_mean_errors.png
  logistic_newton_condition.png
  logistic_predictive.png
  logistic_sd_ratios.png
  logistic_timing_comparison.png
  logistic_vb_gibbs_beta0.png
  logistic_vb_gibbs_beta1.png

Tables saved:
  logistic_diagnostics.csv
  logistic_diagnostics.parquet
